# 把科学观测写成 token，交给同一个 nanoGPT

一本可以整本跑完的教程。选一个科学方向，看它的数据怎样变成一串 token，怎样用同一个两层 nanoGPT 预训练，怎样只用 32 个标签读出答案，最后让程序自己改进模型。

**只用 CPU。** 默认方向在普通笔记本上几分钟跑完，不需要 GPU，不需要下载模型权重。

| | |
|---|---|
| 环境 | Python 3.10 及以上，PyTorch 2.x，JupyterLab 或 Notebook 7 |
| 默认方向 | 扩散场（最快，约 40 秒） |
| 十一个方向 | 光谱、蛋白质、DNA、RNA、小分子、晶体、三维结构、天气场、扩散场、天文图像、文本 |
| 密钥 | 不需要。第 7 节默认用随机搜索，DSK 是可选分支 |

## 目录

| 节 | 你会看到 | 默认方向用时 |
|---|---|---|
| 0 | 选方向，检查依赖 | 几秒 |
| 1 | 数据长什么样，标签怎么分布 | 几秒 |
| 2 | 一条样本怎样切成 token，数值怎样标准化 | 几秒 |
| 3 | 编码器、两层 nanoGPT、读出头各有多少参数 | 几秒 |
| 4 | 预训练损失随数据量下降 | 10 到 30 秒 |
| 5 | 32 个标签训练读出，预测值对真实值 | 几秒 |
| 6 | 和只猜均值、不预训练两条基线比 | 几秒 |
| 7 | 自动研究：程序自己提改动；搜索经验带到没见过的方向 | 看存档几秒，现场 2 轮 1 到 4 分钟 |
| 8 | 星流轨道拟合：调参经验怎样一轮轮改写 | 约 10 秒 |
| 9 | 练习、常见问题、复现说明 | — |

每节开头写着**本节目标**，结尾写着**检查点**。每一节开头的全大写变量都可以改，改完从那一格往下重新运行。十一个方向用同一份代码，只在第 0 节换 \`FIELD\`。

## 先读这一段：这套材料是什么

这套材料的目标是展示一种教学可能性：把不同的科学观测写成 token，交给同一个 nanoGPT，让学员亲眼看到预训练数据量增加时下游误差怎么变。

它不是各领域的真实基准。十一个方向里，五个是公式生成的理想化模拟（光谱、三维结构、天气场、扩散场、天文图像），用来隔离变量、控制难度和 CPU 用时；另外六个用真实数据（蛋白质、DNA、RNA、小分子、晶体、文本）。每个方向用的是什么数据、标签怎么来的、适用范围到哪里，写在仓库的 \`data/eleven/SOURCES.md\`。

有一条贯穿全书的规矩：**方向之间只换编码器和读出方式，两层 nanoGPT 主干完全相同。** 第 3 节会把这条规矩落到代码上，你可以逐行核对。

## 先看结论

这是五个种子、两组留出样本的平均结果。每一行是一个方向在 12,800 个无标签对象上预训练后的下游误差，后面两条基线用来判断这个数算不算好。

| 方向 | 下游任务 | 相对不预训练 | 误差 ÷ 只猜均值 |
|---|---|---|---|
| 光谱 | 紫外有多亮 | 低 41% | 0.20 / 0.21 |
| 蛋白质 | 多久流出来 | 低 9% / 10% | 0.58 / 0.61 |
| DNA | 多少度解开 | 低 21% / 22% | 0.42 / 0.44 |
| RNA | 局部组成多复杂 | 低 22% / 28% | 0.97 / 0.86 |
| 小分子 | 饱和碳占多少 | −7% / +5% | 0.50 / 0.60 |
| 晶体 | 稳不稳定 | 低 25% / 27% | 0.83 / 0.80 |
| 三维结构 | 两端相距多远 | 低 23% / 20% | 0.29 / 0.31 |
| 天气场 | 亮斑有多亮 | 低 26% / 24% | 0.76 / 0.81 |
| 扩散场 | 扩散得多快 | 低 52% / 50% | 0.55 / 0.56 |
| 天文图像 | 星点有多模糊 | 低 14% | 0.72 / 0.70 |
| 文本 | 几个字母被换掉 | 低 31% / 30% | 0.69 / 0.71 |

十一个方向的下游误差在两组留出样本上都随预训练数据量逐档下降。这张表里也有不好看的地方：小分子在 A 组末档比不预训练差 7%，RNA 只比只猜均值好一成左右，晶体和文本的绝对值接近基线。第 9 节会把这几处为什么会出现讲清楚。

完整数字和每个方向的实验约定见 \`docs/pretraining-scale/shared-checkpoints/README.md\`。

## 0. 选方向，检查依赖

**本节目标**：选一个方向，确认环境能跑，并把仓库路径找对。

这一格检查 PyTorch、NumPy、scikit-learn 和 Matplotlib，缺哪个装哪个。在仓库外单独打开这本 Notebook（例如上传到 JupyterLab 或魔搭 Notebook）时，它会先把仓库克隆到当前目录。全程只用 CPU。

选方向的依据：

| 方向 | \`FIELD\` | 数据 | 适不适合第一次看 |
|---|---|---|---|
| 扩散场 | \`field\` | 模拟 | 最快，建议从它开始 |
| 蛋白质 | \`protein\` | 真实 | 标签是实测保留时间，最有“科研感” |
| DNA | \`dna\` | 真实 | 要看全参数微调，稍慢 |
| 文本 | \`text\` | 真实 | 读出方式和其他方向不同，适合讲“读出要配合任务” |
| 其余 | \`spectrum\` \`rna\` \`smiles\` \`crystal\` \`structure3d\` \`weather\` \`image\` | 见第 0 节输出 | 都可以 |

In [ ]:
FIELD = "field"            # 换方向只改这里
SEED = 53                  # 随机种子：决定抽哪 32 个标签、模型的初始权重
N_LABELS = 32              # 下游用多少个带标签的对象。文档里的数字都是 32
THREADS = 2                # CPU 线程数。想复现文档里的数字就保持 2

import importlib.util
import subprocess
import sys
import time
from pathlib import Path

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch",
                           "--index-url", "https://download.pytorch.org/whl/cpu"])
missing = [pip for module, pip in (("numpy", "numpy"), ("sklearn", "scikit-learn"), ("matplotlib", "matplotlib"))
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "nanoscigpt" / "tasks" / "eleven.py").exists()), None)
if root is None:
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/Yu-Yang-Li/nanoSciGPT.git"])
    root = here / "nanoSciGPT"
sys.path.insert(0, str(root))

import numpy as np
import torch
torch.set_num_threads(THREADS)
from nanoscigpt.tasks import eleven, eleven_plots as plots
eleven.N_LABELS = N_LABELS

font = plots.use_chinese_font()
if font is None:
    print("没有找到中文字体，自动下载也没成功，图里的中文会显示成方框。Linux 上可以运行：sudo apt install fonts-noto-cjk")

spec = eleven.FIELDS[FIELD]
print(f"{spec['name']}（{spec['data']}数据）：{spec['question']}。标签：{spec['label']}")
print(f"一个 token = {spec['token']}")
READOUT_NAMES = {"probe": "冻结主干 + 线性头", "finetune": "全参数微调", "probe+surprise": "冻结主干 + 线性头，另加模型的意外程度"}
print(f"下游任务：{spec['task']}；读出：{READOUT_NAMES[spec['readout']]}；"
      f"预训练：{'每档 1,600 步' if spec['budget'] == 'fixed' else '每个对象平均 32 遍'}")

## 1. 看数据

**本节目标**：看清一个方向的数据有几份、各自多少条、标签大致什么范围。

每个方向有三份数据，用途分开：

- **无标签对象**：12,800 个，只有输入，没有答案。预训练只用这一份。
- **标签池**：一两百个带答案的对象，每次从里面抽 \`N_LABELS\` 个（默认 32）来训练下游。
- **留出组 A**：几百个带答案的对象，不参与任何训练，只用来算最后的误差。

下面画出留出组里标签偏低、居中、偏高的三条样本，以及标签的分布。先自己看一眼：凭肉眼能不能从输入猜出标签的高低？

In [ ]:
N_EXAMPLES = 3             # 画几条样本

raw_parts, raw_task = eleven.load(FIELD)
for name, (inputs, _) in {"无标签对象": raw_parts["unlabeled"], "标签池": raw_task["pool"], "留出组 A": raw_task["A"]}.items():
    print(f"{name}：" + "，".join(f"{key} {tuple(values.shape)}" for key, values in inputs.items()))
plots.show_samples(FIELD, raw_task, N_EXAMPLES)

**检查点**：图上方的粗体句子应该说出这个方向的标签范围和极端样本。如果你看不出输入和标签有什么关系，记下来——第 6 节会告诉你模型能读到多少。

## 2. 预处理：把一个对象切成 token

**本节目标**：看一条样本怎样变成模型真正读到的数字，以及为什么数值还要标准化。

语言模型读的是一串 token。这一步把对象写成这样一串：左图是原始对象和 token 的边界，右图是模型实际读到的内容。

- 符号类输入（蛋白质、DNA、RNA、小分子、文本）：一个符号一个 token，查词表变成编号。
- 数值类输入（光谱、图像、各种场、距离矩阵）：相邻的几个数组成一个 token。
- 晶体：晶格矩阵是第一个 token，之后每个原子一个 token。

改 \`ROW\` 可以换一条样本看。

In [ ]:
ROW = 0                    # 看留出组里的第几条样本

plots.show_tokens(FIELD, raw_task, ROW)
eleven.show_tokens(FIELD, {name: values[ROW] for name, values in raw_task["A"][0].items()})
print("这条样本的下游标签：", round(float(raw_task["A"][1][ROW]), 4))

数值类输入还要做标准化：减去均值、除以标准差，让不同位置的数在同一个量级上。均值和标准差只从抽中的那些带标签的训练对象上算，评测数据不参与，这样留出组的信息不会漏进训练。符号类输入不需要这一步，下面改为画符号的出现次数。

这一步看着琐碎，但它决定了曲线可不可信：如果标准化用了留出组，后面的下降就是假的。

In [ ]:
parts, task = eleven.normalize(FIELD, raw_parts, raw_task, SEED)
plots.show_normalization(FIELD, raw_task, task)

**检查点**：右图的数值分布应该集中在 0 附近、幅度在 1 上下；符号类输入则是一张出现次数条形图。

## 3. 搭模型

**本节目标**：确认十一个方向共用同一个主干，只换编码器和读出头。

模型分三段：

1. **编码器**：把每个 token 变成一个 32 维的向量。十一个方向只有这一段不同，共三种（符号、数值块、晶胞）。
2. **两层 nanoGPT**：十一个方向结构完全相同。每个位置只能看到它前面的 token。
3. **读出头**：一个线性层，把整条序列的表示变成一个数，也就是下游任务的答案。

下面打印编码器的结构，再画出每一段有多少参数。把 \`COMPARE_WITH\` 换成别的方向，可以看到两个方向只有编码器和位置编码的参数个数不同，主干部分一模一样。

In [ ]:
COMPARE_WITH = "protein"   # 拿哪个方向来对比参数个数

model = eleven.network(FIELD)
print(model.encoder)
print(f"主干：{len(model.blocks)} 层，每层 {model.blocks[0].attn.n_head} 个注意力头，向量 {model.blocks[0].attn.n_embd} 维，"
      f"序列最长 {model.wpe.num_embeddings} 个 token")
plots.show_model([FIELD] if COMPARE_WITH == FIELD else [FIELD, COMPARE_WITH])

**检查点**：两个方向的参数条形图里，注意力和 LayerNorm 那几段应该一样长，只有编码器和位置编码不同。代码里 \`nanoscigpt/scientific/scigpt.py\` 的构造只接收编码器和位置数，主干来自仓库的 nanoGPT。

## 4. 预训练：只用无标签数据

**本节目标**：看预训练损失怎样随无标签数据量下降，以及训练损失和留出损失差多少。

预训练的题目是“根据前面的 token 预测下一个 token”，答案就在数据自己里面，不需要任何标签。这里用不同数量的无标签对象各训练一次，看数据多了以后模型是不是学得更好。

- 曲线：训练过程中每一批数据上的损失。
- 圆点：训练结束后，在留出样本（没参加训练）上的损失。圆点比曲线末端高，说明模型记住了一部分训练数据。

把 12800 加进 \`SIZES\` 可以跑到最大档，用时大约是现在的四倍。

In [ ]:
SIZES = [200, 800, 3200]   # 预训练用的无标签对象数，可以加上 12800

models = {0: eleven.pretrain(FIELD, parts, 0, SEED)}   # 0 = 不预训练，作对照
curves, pretrain_loss = {}, {}
for size in SIZES:
    start = time.perf_counter()
    curves[size] = []
    models[size] = eleven.pretrain(FIELD, parts, size, SEED, curve=curves[size])
    pretrain_loss[size] = eleven.heldout_loss(models[size], parts["A"][0])
    print(f"{size:>6} 个无标签对象：更新 {len(curves[size]):>5} 次，留出样本上的损失 {pretrain_loss[size]:.4f}，用时 {time.perf_counter() - start:.0f} 秒")
plots.show_training(curves, pretrain_loss)

**检查点**：右图的三颗圆点应该从左到右往下走。如果中间某一档反而升高，多半是种子或步数的问题，换 \`SEED\` 再看一次。

## 5. 下游训练：用少量标签学会回答问题

**本节目标**：看 32 个标签能训练出什么，以及预训练到底帮了多少。

现在才用到标签。从标签池里抽 \`N_LABELS\` 个对象，接在预训练好的模型后面训练读出。读出方式有三种，每个方向有自己的默认值：

- **冻结主干 + 线性头**：主干不动，只训练最后那个线性层。
- **全参数微调**：所有权重一起更新 200 步。
- **冻结主干 + 线性头，另加模型的意外程度**：只用于文本。除了表示，再给线性头两个数：模型逐个预测下一个字母时，平均有多意外、最意外的一处有多意外。拼写学得越熟的模型，碰到被换掉的字母越意外。

每个预训练规模各训练一次读出，再加一个“不预训练”的对照：同样的模型、同样的标签，只是主干没有预训练过。图里每个点是留出组的一个对象，越贴近对角线越好。

In [ ]:
READOUT = spec["readout"]  # "probe"：冻结主干 + 线性头；"finetune"：全参数微调；"probe+surprise"：另加意外程度（文本）
COMPARE = (0, max(SIZES))  # 画哪两个模型的预测。0 是不预训练

downstream_error, predictions = {}, {}
for size, trained in models.items():
    predictions[size] = []
    downstream_error[size], guess_mean = eleven.downstream(FIELD, trained, task, SEED, predictions=predictions[size], readout=READOUT)
    name = "不预训练" if size == 0 else f"{size} 个对象预训练"
    print(f"{name:>14}：留出组 A 上的平均绝对误差 {downstream_error[size]:.4f}")
print(f"{'只猜平均值':>14}：{guess_mean:.4f}")
plots.show_predictions(task["A"][1], {("不预训练" if size == 0 else f"{size:,} 个对象预训练"): np.array(predictions[size]) for size in COMPARE},
                       guess_mean, spec["task"])

**检查点**：两张散点图里，预训练之后的那张应该更贴近对角线。如果两张差不多，说明在这个方向上 32 个标签和这套读出已经到顶了。

## 6. 评测

**本节目标**：把误差放到两条基线旁边看，再和五个种子的结论对照。

误差要和基线比才有意义。这里有两条基线：

- **只猜平均值**：不用任何模型，对每个对象都回答训练标签的平均值。比它还差，说明模型没学到东西。
- **不预训练**：检验预训练有没有帮上忙。

左图是下游误差随预训练数据量的变化，和两条基线画在一起。右图是预训练损失。要看的是两条曲线是不是随数据量一起下降，以及下游误差落在两条基线的哪一边。

这里只跑了一个种子、到 3,200 档，曲线会有起伏。下一格读入五个种子、四档、两组留出的汇总，和刚才的数字对一下。

In [ ]:
plots.show_scaling(pretrain_loss, downstream_error, guess_mean, f"{spec['name']} · {spec['question']}", spec["task"])
best = min(SIZES, key=downstream_error.get)
print(f"最好的一档是 {best:,} 个对象：误差 {downstream_error[best]:.4g}，"
      f"比不预训练{'低' if downstream_error[best] < downstream_error[0] else '高'} {abs(1 - downstream_error[best] / downstream_error[0]):.0%}，"
      f"比只猜平均值{'低' if downstream_error[best] < guess_mean else '高'} {abs(1 - downstream_error[best] / guess_mean):.0%}。")

下面这张表是仓库里封存的五个种子结果，两组留出样本各一次。你自己的单个种子结果会和它有小幅差别，这是正常的。

In [ ]:
import csv

summary = root / "docs/pretraining-scale/shared-checkpoints/summary.csv"
with summary.open(encoding="utf-8-sig") as handle:
    rows = [row for row in csv.DictReader(handle) if row["方向"] == spec["name"]]
for row in rows:
    values = " / ".join(f"{float(row['下游' + n]):.4g}" for n in ("200", "800", "3200", "12800"))
    print(f"留出组 {row['组']}，{row['读出']}：{values}")
    print(f"  只猜均值 {float(row['只猜均值']):.4g}，不预训练 {float(row['下游不预训练']):.4g}，"
          f"逐档下降 {row['下游下降段']}/3 段，端点改善 {row['下游端点改善种子']}/5 个种子")

**检查点**：五种子表的末档应该和你的单种子结果在同一个量级，误差是只猜均值的零点几倍。如果相差一倍以上，先确认 \`THREADS\` 是 2、种子是 53。

## 7. 自动研究：让程序自己提改动、跑实验

**本节目标**：看一个搜索循环怎样在主干不变的前提下改编码方式和训练设置，以及经验能不能从见过的方向带到没见过的方向。

前面每个设置都是人定的。这一节把“改设置、跑实验、看结果、再改”交给程序。十一个方向用同一套规则：

- **不能改**：两层 nanoGPT 主干。
- **可以改**：相邻两个 token 要不要并成一个（\`merge\`）、预训练的题目（预测下一个、隔一个预测、先遮住一部分再预测）、预训练步数和学习率、读出方式和它的步数、学习率、岭回归强度、表示的汇总方式。
- **怎么打分**：每个配置在开发集上评分。开发集是标签池里没被抽去训练的对象。留出组 A 只在搜索结束后用一次，评开发集上最好的那个配置。
- **两种提改动的办法**：DSK（一个大语言模型）读完此前每一轮的结果再提下一个配置；随机搜索在同样的范围里随便抽。两边轮数相同。

先看存档的结果：每个方向 DSK 和随机搜索各跑 2 到 4 次，每次 8 轮。跑不赢随机搜索的方向照实标出。

In [ ]:
from autoresearch import common_search as search, search_plots

runs = [run for run in search.archived(FIELD) if run["method"] != "skill"]
search_plots.show_search(FIELD, runs)
for run in runs:
    if run["method"] == "model" and run["repeat"] == 1:
        print("DSK 第 1 次运行每一轮的理由：")
        for trial in run["trials"][1:]:
            print(f"  第 {trial['round']} 轮，开发集误差 {trial['dev']:.4f}：{trial['reason'][:90]}")

十一个方向放在一起看：

In [ ]:
search_plots.show_all_fields()

下面现场跑一次。默认用随机搜索，不需要任何密钥。想让 DSK 来提改动，先在启动 Jupyter 之前设置环境变量 \`SCNET_API_BASE\`、\`SCNET_API_KEY\`、\`SCNET_MODEL\`，再把 \`METHOD\` 改成 "model"。密钥只从环境变量读，不要写进 Notebook。

默认只跑 2 轮、1 个种子，为的是几分钟内看完流程。这么少的轮数很可能找不到比起点更好的配置，结果的波动也比存档大。想认真搜索就把轮数和种子调到存档的设置，用时会长到几十分钟。运行记录存在仓库的 \`out/\` 目录下，同一格重新运行会接着上次的进度；换 \`DEV_SEEDS\` 之前请先删掉那里的记录。

In [ ]:
METHOD = "random"          # "random" 随机搜索；"model" 让 DSK 提改动（需要环境变量里有密钥）
ROUNDS = 2                 # 轮数。存档用的是 8
DEV_SEEDS = (53,)          # 每个配置用几个种子打分。存档用的是 (53, 71, 89)

journal = search.run(FIELD, METHOD, root / "out" / f"search-{N_LABELS}labels", rounds=ROUNDS, dev_seeds=DEV_SEEDS, confirm_seeds=DEV_SEEDS)
search_plots.show_search(FIELD, [journal])

### 7.1 把搜索经验带到没见过的方向

上面的搜索每个方向都从零开始，在一个方向上学到的东西带不到下一个方向。这一小节试着把它留下来：

1. 十一个方向分成 4 折，轮着留出。
2. 每一折，DSK 读其余方向的全部搜索记录，写成一份一千多字的搜索经验：哪类输入先试什么、哪些改动通常没用、8 轮怎么分配。
3. 经验锁定后，带到这一折留出的方向上搜索。提示里只多了这份经验，轮数、种子、打分方式都和之前一样。
4. 和不带经验的 DSK、同预算的随机搜索比。

留出哪几个方向、怎么判定“有用”，都在运行之前写定。左图按“此前的搜索有没有找到改进”把方向分成两组；右图看搜索得快不快。

In [ ]:
from autoresearch import search_skill

search_plots.show_skill_transfer()

下面打印当前方向用到的那份经验。写它的时候，DSK 没有见过这个方向的任何记录。

In [ ]:
print(f"{spec['name']}在 {search_skill.fold_of(FIELD)} 折，这一折留出的方向："
      + "、".join(eleven.FIELDS[field]["name"] for field in search_skill.FOLDS[search_skill.fold_of(FIELD)]))
print(search_skill.locked(FIELD))
search_plots.show_search(FIELD, search.archived(FIELD))

**检查点**：存档图里每个方向的 DSK 和随机搜索应该挨着画，胜负按事先写定的标准标出。带经验的曲线不一定更高，这里要如实看。

## 8. 星流轨道拟合：调参智能体怎样积累经验

**本节目标**：看“经验”在另一个任务上是什么形态，以及它为什么会先变好再变差。

上一节的程序每次都从零开始，上一次搜索学到的东西不会留给下一次。这一节换一个任务，看“把经验留下来”是什么样子。

任务是拟合银河系里一条恒星流（GD-1）的轨道：给定一组观测，调 7 个参数，让算出来的轨道穿过观测点。智能体是 DSK，一次拟合只能试 40 组参数。实验分两层：

- **一次拟合之内**：DSK 每交 10 组参数，看到拟合得好不好，再交下一批。
- **多次拟合之间**：DSK 把调参心得写成一份一千多字的经验。每轮先带着经验拟合两个训练场景，对照答案改写经验，然后锁定，拿去拟合 8 个从没见过的场景。共 8 轮。

这一节的图全部来自存档的参数，轨道在你的电脑上重新积分，不需要密钥。先看一个没见过的场景：每一轮的经验各拟合一次，找到的最好轨道叠在观测点上。

In [ ]:
SCENARIO = 9005            # 盲测场景，可选 9001 到 9008
SHOW_ROUNDS = (0, 1, 2, 5, 8)   # 画哪几轮的经验。0 是空经验

from autoresearch import stream_plots

stream_plots.show_orbits(SCENARIO, SHOW_ROUNDS)

一次拟合之内发生了什么：上排是空经验，下排是第 8 轮的经验，颜色从深到浅是第 1 到第 4 次提交之后的最好轨道。

In [ ]:
stream_plots.show_one_fit(SCENARIO, rounds=(0, 8))

经验是怎么被改写的。

- 第一张图：九条有代表性的规则，在每一轮的经验里有没有。能看到有的规则一开始就有并一直留着，有的被证据推翻后删掉（“围绕一个中心点撒点”），有的后来才加进来。
- 第二张图：经验的长度、每次改写保留了上一轮多少文字、每一轮的规则注明依据的是哪几轮的训练场景。

图下面是 DSK 自己写的每轮改动说明。想读某一轮经验的全文，把 \`READ_ROUND\` 改成那一轮的编号。

In [ ]:
READ_ROUND = None          # 填 1 到 8 的数字，打印那一轮经验的全文

stream_plots.show_skill_changes()
if READ_ROUND:
    print(stream_plots.skill_text(READ_ROUND))

最后看成绩。纵轴是在没见过的场景上，找到的最好轨道离答案还有多远，越低越好。

读这张图要注意三件事：带着经验比空经验好；最好的是第 5 轮，之后三轮又慢慢变差，没有一轮比一轮好下去；和同样预算的 Nelder–Mead（一种不用模型的经典优化方法）相比，优势不明显。经验的完整文本、实验规则和后续复验见仓库的 \`docs/stream-skill.md\`。

In [ ]:
stream_plots.show_blind_scores()

## 9. 练习与延伸

**本节目标**：把笔记本里的设置改成自己的问题，验证结论是不是还成立。

- 回到第 0 节换一个 \`FIELD\`，从头再跑一遍，比较两个方向的图。
- 第 0 节把 \`N_LABELS\` 改成 8 或 128，从头再跑：标签更少时，预训练的帮助是更大还是更小？
- 第 5 节换一种 \`READOUT\`：冻结主干和全参数微调，哪一种在这个方向上更好？
- 第 7 节把 \`ROUNDS\` 和 \`DEV_SEEDS\` 调到存档的设置，看你的随机搜索和存档里的差多少。
- 命令行里做同样的事：\`python -m nanoscigpt.tasks.eleven --field field\` 和 \`python -m autoresearch.common_search --field field --method random\`。

十一个方向里，有三个方向可以当反例讲：

- **RNA**：标签是三核苷酸熵，能从序列直接数出来，所以末档只比只猜均值好一成。
- **DNA**：预训练读基因组片段，下游是人工发夹，两者不完全对应，预训练损失到 3,200 档就不再降。
- **晶体**：只有 32 个标签，200、800 两档还不如只猜均值，到 3,200 档才超过。

这三个反例说明的是同一件事：预训练不是万能的，标签的数量、难度和它与数据的对应关系，决定了预训练有没有空间。

## 常见问题

**跑得慢。** 第 4 节的预训练占了大头。把 \`SIZES\` 改成 \`[200, 800]\` 先看流程；\`THREADS\` 调到 4 可以快一些，但会改变数值细节。

**图里的中文是方框。** 系统缺中文字体。第 0 节会尝试下载 Noto Sans SC；离线环境请先装 \`fonts-noto-cjk\`（Debian/Ubuntu）或在系统里装一款中文字体，再重启内核。

**在魔搭 Notebook 里打开。** 直接把仓库克隆到一个工作目录，再打开 \`notebooks/eleven_fields.ipynb\`。第 0 节会自己补装缺少的依赖；镜像通常自带 PyTorch，这一步会很快跳过。整本跑完默认方向约两到三分钟，选 DNA 会久一些。

**想用 DSK 提改动。** 在启动 Jupyter 之前设好 \`SCNET_API_BASE\`、\`SCNET_API_KEY\`、\`SCNET_MODEL\` 三个环境变量，再把第 7 节的 \`METHOD\` 改成 \`"model"\`。密钥只从环境变量读。

**结果和文档里的数字差一点。** 文档是五个种子的平均。这里是单种子、三档，曲线有起伏是正常的；Linux 和 Windows 的浮点顺序不同，也会有百分之一到二十分之一的差别。

## 复现说明

- **数据**：六个真实数据方向的文件在 \`data/eleven/\`，来源和许可写在 \`data/eleven/SOURCES.md\`。五个模拟方向按固定随机流现场生成，同一台机器上逐位可复现。
- **随机性**：种子 53 决定预训练顺序、初始权重和抽哪 32 个标签。想复现文档里种子 53 那一列，保持 \`SEED = 53\`、\`THREADS = 2\`。
- **五个种子的完整表**：\`docs/pretraining-scale/shared-checkpoints/README.md\`，逐组留出结果在同目录的 \`all-holdout-results.csv\`。
- **首次使用检查**：Windows 和 WSL 从空目录安装、整本执行、逐方向跑命令行的记录在 \`docs/first-run.md\`。
- **版本**：V2 定稿的复核记录在 \`docs/version-v2.md\`。

如果这本 Notebook 帮你省了时间，欢迎在仓库里提 issue 说一说你换成了哪个方向、看到了什么。